# M3_LS: roberta-large plus label smoothing

Standalone version of the label-smoothing run. It first trains M3 (klue/roberta-large + markers), then continues from the best checkpoint for 3 more epochs with label smoothing (alpha 0.1).

Label smoothing trades a small micro-F1 gain for lower AUPRC because it flattens the predicted probabilities. Both numbers print at the end so the trade-off is visible.

Config: M3 at lr 1e-5 for 5 epochs, then LS for 3 epochs. Results land in `results/M3_LS/`.

In [2]:
import os, json, random, shutil, warnings
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy.special

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    EarlyStoppingCallback, DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, average_precision_score, confusion_matrix
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")


In [3]:
SEED = 42
MAX_LENGTH = 256
NUM_LABELS = 30
NO_RELATION_ID = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RESULTS_DIR = os.path.join(BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("cuda", torch.cuda.is_available(), "devices", torch.cuda.device_count())

cuda True devices 2


In [4]:
ds = load_dataset("klue/klue", "re")
train_ds, val_ds = ds["train"], ds["validation"]
label_names = val_ds.features["label"].names
val_labels = np.array(val_ds["label"])
print("train", len(train_ds), "val", len(val_ds), "labels", len(label_names))

README.md: 0.00B [00:00, ?B/s]

re/train-00000-of-00001.parquet:   0%|          | 0.00/6.65M [00:00<?, ?B/s]

re/validation-00000-of-00001.parquet:   0%|          | 0.00/1.54M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32470 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7765 [00:00<?, ? examples/s]

train 32470 val 7765 labels 30


In [5]:
def add_markers(example):
    text = example["sentence"]
    subj, obj = example["subject_entity"], example["object_entity"]
    spans = [
        (subj["start_idx"], subj["end_idx"], subj["type"], "subj"),
        (obj["start_idx"], obj["end_idx"], obj["type"], "obj"),
    ]
    spans.sort(key=lambda s: s[0], reverse=True)
    for start, end, etype, role in spans:
        word = text[start:end + 1]
        tag = "@*" + etype + "*" + word + "@" if role == "subj" else "#^" + etype + "^" + word + "#"
        text = text[:start] + tag + text[end + 1:]
    return text


def micro_f1(labels, preds):
    mask = labels != NO_RELATION_ID
    if mask.sum() == 0:
        return 0.0
    return float(f1_score(labels[mask], preds[mask], average="micro", zero_division=0))


def auprc(labels, logits):
    probs = scipy.special.softmax(logits, axis=1)
    onehot = label_binarize(labels, classes=list(range(NUM_LABELS)))
    return float(average_precision_score(onehot, probs, average="micro"))


def metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"micro_f1": micro_f1(labels, preds), "auprc": auprc(labels, logits)}

In [6]:
def save_results(name, trainer, f1, ap, best_epoch, labels, preds):
    out = os.path.join(RESULTS_DIR, name)
    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, "metrics.json"), "w") as f:
        json.dump({"model": name, "micro_f1": f1, "auprc": ap, "best_epoch": best_epoch},
                  f, indent=2)

    cm = confusion_matrix(labels, preds, labels=list(range(NUM_LABELS)))
    short = [n[:10] for n in label_names]
    fig, ax = plt.subplots(figsize=(16, 14))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.03)
    ax.set_xticks(range(NUM_LABELS)); ax.set_xticklabels(short, rotation=90, fontsize=6)
    ax.set_yticks(range(NUM_LABELS)); ax.set_yticklabels(short, fontsize=6)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name + " confusion matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(out, "confusion_matrix.png"), dpi=120, bbox_inches="tight")
    plt.close()

    logs = trainer.state.log_history
    tl = [(l["epoch"], l["loss"]) for l in logs if "loss" in l and "eval_loss" not in l]
    vl = [(l["epoch"], l["eval_loss"]) for l in logs if "eval_loss" in l]
    vf = [(l["epoch"], l["eval_micro_f1"]) for l in logs if "eval_micro_f1" in l]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    if tl: ax[0].plot(*zip(*tl), "b-o", ms=3, label="train_loss")
    if vl: ax[0].plot(*zip(*vl), "r-o", ms=3, label="val_loss")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].grid(alpha=0.3)
    if vf: ax[1].plot(*zip(*vf), "g-o", ms=3, label="val_micro_f1")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("micro_f1"); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out, "loss_curve.png"), dpi=120, bbox_inches="tight")
    plt.close()
    print(name, "written to", out)

In [7]:
MODEL = "klue/roberta-large"
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(example):
    enc = tok(add_markers(example), max_length=MAX_LENGTH, truncation=True)
    enc["labels"] = example["label"]
    return enc

train_enc = train_ds.map(encode, remove_columns=train_ds.column_names)
val_enc = val_ds.map(encode, remove_columns=val_ds.column_names)

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Map:   0%|          | 0/32470 [00:00<?, ? examples/s]

Map:   0%|          | 0/7765 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
warmup = int(((len(train_enc) // 16) // 2) * 5 * 0.1)

args = TrainingArguments(
    output_dir=os.path.join(BASE, "m3_ckpt"),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=warmup,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    fp16=True,
    seed=SEED,
    data_seed=SEED,
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_enc, eval_dataset=val_enc,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[checkpoint load: klue/roberta LayerNorm gamma/beta renamed to weight/bias; classifier head newly initialised]


In [9]:
trainer.train()
pred = trainer.predict(val_enc)
m3_f1 = micro_f1(val_labels, np.argmax(pred.predictions, axis=-1))
print("M3 baseline micro_f1", round(m3_f1, 4))

Epoch,Training Loss,Validation Loss,Micro F1,Auprc
1,3.670243,2.227659,0.580408,0.684141
2,2.376231,1.515477,0.664965,0.824311
3,1.692085,1.378012,0.693044,0.845291
4,1.267074,1.424396,0.724633,0.853897
5,0.938492,1.453959,0.729100,0.855440


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[checkpoint load: klue/roberta LayerNorm gamma/beta renamed to weight/bias; classifier head newly initialised]


M3 baseline micro_f1 0.7291


In [10]:
from torch.nn import CrossEntropyLoss

class LabelSmoothingTrainer(Trainer):
    def __init__(self, *args, label_smoothing=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.label_smoothing = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = CrossEntropyLoss(label_smoothing=self.label_smoothing)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

warmup_ls = int(((len(train_enc) // 16) // 2) * 3 * 0.1)
args_ls = TrainingArguments(
    output_dir=os.path.join(BASE, "m3ls_ckpt"),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=warmup_ls,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    fp16=True,
    seed=SEED,
    data_seed=SEED,
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = LabelSmoothingTrainer(
    model=model, args=args_ls,
    train_dataset=train_enc, eval_dataset=val_enc,
    processing_class=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics_fn, label_smoothing=0.1,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [11]:
trainer.train()
pred = trainer.predict(val_enc)
preds = np.argmax(pred.predictions, axis=-1)
f1 = micro_f1(val_labels, preds)
ap = auprc(val_labels, pred.predictions)
best_epoch = max((l for l in trainer.state.log_history if "eval_micro_f1" in l),
                 key=lambda x: x["eval_micro_f1"])["epoch"]
print("M3_LS", "micro_f1", round(f1, 4), "auprc", round(ap, 4), "best_epoch", best_epoch)
print("label smoothing delta vs M3:", round(f1 - m3_f1, 4),
      "| auprc moved to", round(ap, 4))
save_results("M3_LS", trainer, f1, ap, best_epoch, val_labels, preds)

Epoch,Training Loss,Validation Loss,Micro F1,Auprc
1,1.656217,1.335456,0.727186,0.795898
2,1.556545,1.291392,0.707722,0.813930
3,1.465764,1.325482,0.685705,0.806103
4,1.419593,1.386852,0.723995,0.780954


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[checkpoint load: klue/roberta LayerNorm gamma/beta renamed to weight/bias; classifier head newly initialised]


M3_LS micro_f1 0.7272 auprc 0.7956 best_epoch 1.0
label smoothing delta vs M3: -0.0019 | auprc moved to 0.7956
M3_LS written to /kaggle/working/results/M3_LS
